# Full Atlas-Free ALE3DCNN Text-to-Brain Pipeline

This notebook implements the staged atlas-free CNN pipeline that is the closest CNN analogue of the pretrained NeuroVLM MLP text-to-brain workflow.

Stages:

1. CNN autoencoder: `ALE volume -> ALE3DCNNEncoder -> 384 latent -> ALE3DCNNDecoder -> reconstructed ALE volume`
2. Text-to-CNN-latent training: `SPECTER -> text projection -> CNN latent -> frozen CNN decoder -> reconstructed/generated ALE volume`
3. Contrastive alignment: `CNN encoder(ALE) <-> text projection(SPECTER)` using symmetric InfoNCE
4. Text-to-brain generation: `text -> SPECTER -> text projection -> CNN decoder -> generated atlas-free ALE volume`

It saves retrieval metrics, generation metrics, checkpoint artifacts, and qualitative true-vs-generated maps.

## 1. Runtime, Dependencies, Repo

In [ ]:
import os, platform, subprocess, sys, time, json, shutil, traceback, copy
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)

!pip -q install nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn

REPO_URL = os.environ.get('NEUROVLM_REPO_URL', 'https://github.com/neurovlm/neurovlm.git')
REPO_BRANCH = os.environ.get('NEUROVLM_REPO_BRANCH', 'neurovlm_gnn')
REPO_DIR = os.environ.get('NEUROVLM_REPO_DIR', '/content/neurovlm_gnn')
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))
sys.path.insert(0, str(Path(REPO_DIR)))
print('Working directory:', os.getcwd())

## 2. Drive and Config

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

MODEL_NAME = 'ale_3dcnn_full_pipeline'
DRIVE_ROOT = '/content/drive/MyDrive/neurovlm'
DRIVE_DATA_DIR = f'{DRIVE_ROOT}/data_ale_3dcnn'
DRIVE_CACHE_DIR = f'{DRIVE_DATA_DIR}/ale_caches'
LOCAL_CACHE_DIR = '/content/ale_caches_ale_3dcnn'
RUNS_DIR = f'{DRIVE_ROOT}/runs_{MODEL_NAME}'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
COMPARISON_FILE = f'{RUNS_DIR}/full_pipeline_comparison.csv'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')

# Preprocessing: match current atlas-free ALE3DCNN.
RESOLUTION_MM = 4.0
KERNEL_FWHM_MM = 9.0
CROP_TO_BRAIN = True
NORMALIZE = 'max'
CLAMP = True
CACHE_DTYPE = 'float16'

# Model.
LATENT_DIM = 384
BASE_CHANNELS = 48
NUM_BLOCKS = 4
DROPOUT = 0.1
NORM = 'group'
POOLING = 'max'
SEED = 42

# Stage 1 autoencoder.
TRAIN_AUTOENCODER_IF_MISSING = True
AE_BATCH_SIZE = 64
AE_EPOCHS = 100
AE_LR = 3e-4
AE_WEIGHT_DECAY = 1e-4
AE_LOSS = 'mse'  # 'mse' or 'bce_logits'

# Stage 2 text -> CNN latent -> decoder.
TRAIN_TEXT_DECODER_IF_MISSING = True
TEXT_DECODER_INITS = ['pretrained_infonce', 'random']
TEXT_DECODER_EPOCHS = 100
TEXT_DECODER_BATCH_SIZE = 256
TEXT_DECODER_LR = 1e-4
TEXT_DECODER_WEIGHT_DECAY = 1e-4
TEXT_DECODER_LOSS = AE_LOSS

# Stage 3 contrastive.
CONTRASTIVE_EPOCHS = 150
CONTRASTIVE_BATCH_SIZE = 512
CONTRASTIVE_BATCH_SIZE_CANDIDATES = [1024, 512, 384, 256, 128]
LR_CNN = 1e-4
LR_PROJ = 1e-5
TEMPERATURE = 0.07
VAL_INTERVAL = 5
EARLY_STOPPING_PATIENCE = 25
TRAIN_SANITY_N = 512
NUM_WORKERS = 4

for d in [DRIVE_ROOT, DRIVE_DATA_DIR, DRIVE_CACHE_DIR, LOCAL_CACHE_DIR, RUNS_DIR, EVAL_RESOURCE_DIR]:
    os.makedirs(d, exist_ok=True)

CACHE_BASENAME = f'atlas_free_ale_{int(RESOLUTION_MM)}mm_fwhm{str(KERNEL_FWHM_MM).replace(".", "p")}_crop_{CACHE_DTYPE}.pt'
ATLAS_FREE_CACHE = f'{LOCAL_CACHE_DIR}/{CACHE_BASENAME}'
DRIVE_ATLAS_FREE_CACHE = f'{DRIVE_CACHE_DIR}/{CACHE_BASENAME}'
for candidate in [
    DRIVE_ATLAS_FREE_CACHE,
    f'{DRIVE_CACHE_DIR}/atlas_free_{int(RESOLUTION_MM)}mm_fwhm{int(KERNEL_FWHM_MM)}_crop_{CACHE_DTYPE}.pt',
    f'{DRIVE_CACHE_DIR}/atlas_free_{RESOLUTION_MM:g}mm_fwhm{KERNEL_FWHM_MM:g}_crop_{CACHE_DTYPE}.pt',
]:
    if os.path.exists(candidate) and not os.path.exists(ATLAS_FREE_CACHE):
        shutil.copy2(candidate, ATLAS_FREE_CACHE)
        print('Copied Drive cache to local:', candidate, '->', ATLAS_FREE_CACHE)
        break

CONFIG = {k: v for k, v in globals().items() if k.isupper() and k not in {'CONFIG'}}
with open(Path(RUNS_DIR) / f'full_pipeline_config_{RUN_STAMP}.json', 'w') as f:
    json.dump({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, f, indent=2)
print('Runs:', RUNS_DIR)
print('Cache:', ATLAS_FREE_CACHE)

## 3. Dataset and Imports

In [ ]:
import argparse, importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score

from neurovlm.gnn.ale_cnn import ALE3DCNNAutoEncoder, ALE3DCNNEncoder, ALE3DCNNDecoder, count_parameters
from neurovlm.gnn.ale_dataset import ALEPreprocessConfig, ALEVolumeDataset, build_or_load_ale_cache
from neurovlm.gnn.model import TextProjHead
from neurovlm.loss import InfoNCELoss

TRAIN_MODULE_PATH = str(Path(REPO_DIR) / 'experiments' / 'train_ale_cnn.py')
_spec = importlib.util.spec_from_file_location('train_ale_cnn_colab', TRAIN_MODULE_PATH)
train_ale_cnn = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(train_ale_cnn)

device = train_ale_cnn.which_device('cuda' if torch.cuda.is_available() else 'auto')
print('Device:', device)
torch.manual_seed(SEED)
np.random.seed(SEED)

dataset_args = argparse.Namespace(
    mode='atlas_free', kernel_fwhm_mm=KERNEL_FWHM_MM, resolution_mm=RESOLUTION_MM,
    crop_to_brain=CROP_TO_BRAIN, normalize=NORMALIZE, no_clamp=not CLAMP,
    cache_dtype=CACHE_DTYPE, cache_file=ATLAS_FREE_CACHE, force_rebuild_cache=False,
    max_papers=None, run_dir=str(Path(RUNS_DIR) / f'dataset_split_{RUN_STAMP}'),
    seed=SEED, val_frac=0.1, test_frac=0.1,
)
ds, train_ds, val_ds, test_ds, cache_payload, preprocess_config = train_ale_cnn.build_dataset(dataset_args)
print('Input shape:', ds.input_shape)
print('Splits:', len(train_ds), len(val_ds), len(test_ds))
if os.path.exists(ATLAS_FREE_CACHE) and not os.path.exists(DRIVE_ATLAS_FREE_CACHE):
    shutil.copy2(ATLAS_FREE_CACHE, DRIVE_ATLAS_FREE_CACHE)
    print('Copied built cache to Drive:', DRIVE_ATLAS_FREE_CACHE)

## 4. Shared Helpers

In [ ]:
def make_volume_loader(split_ds, batch_size, shuffle):
    return DataLoader(
        split_ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )

def reconstruction_loss(logits, target, loss_name):
    target = target.float()
    if loss_name == 'mse':
        return F.mse_loss(logits, target)
    if loss_name == 'bce_logits':
        return F.binary_cross_entropy_with_logits(logits, target.clamp(0, 1))
    raise ValueError(loss_name)

def display_volume(logits, loss_name):
    return torch.sigmoid(logits) if loss_name == 'bce_logits' else logits.clamp(0, 1)

def make_text_projector(init, freeze=False):
    if init == 'pretrained_infonce':
        from neurovlm.models import ProjHead
        text_proj = copy.deepcopy(ProjHead.from_pretrained('text_infonce'))
    elif init == 'random':
        text_proj = TextProjHead(in_dim=768, hidden_dim=512, out_dim=LATENT_DIM)
    else:
        raise ValueError(init)
    for p in text_proj.parameters():
        p.requires_grad_(not freeze)
    return text_proj

def build_cnn_encoder(load_state=None, freeze=False):
    enc = ALE3DCNNEncoder(
        base_channels=BASE_CHANNELS, num_blocks=NUM_BLOCKS, out_dim=LATENT_DIM,
        dropout=DROPOUT, norm=NORM, pooling=POOLING,
    )
    if load_state is not None:
        enc.load_state_dict(load_state)
    for p in enc.parameters():
        p.requires_grad_(not freeze)
    return enc

def latest_existing(paths):
    existing = [Path(p) for p in paths if Path(p).exists()]
    return str(max(existing, key=lambda p: p.stat().st_mtime)) if existing else None

def find_latest_checkpoint(patterns):
    hits = []
    for pattern in patterns:
        hits.extend(Path('/').glob(pattern.lstrip('/')) if pattern.startswith('/') else Path('.').glob(pattern))
    hits = [p for p in hits if p.exists()]
    return str(max(hits, key=lambda p: p.stat().st_mtime)) if hits else None

@torch.no_grad()
def volume_pair_metrics(model_fn, split_ds, *, split_name, loss_name, batch_size=64, top_fracs=(0.01, 0.05, 0.10), target_threshold=1e-6):
    loader = make_volume_loader(split_ds, batch_size, shuffle=False)
    mse_sum = 0.0
    mae_sum = 0.0
    voxel_count = 0
    bce_losses = []
    map_pearsons = []
    voxel_aurocs = []
    voxel_average_precisions = []
    top_dice = {f'top_{int(frac * 100)}pct_dice': [] for frac in top_fracs}

    for batch in loader:
        x = batch['volume'].float().to(device, non_blocking=True)
        logits = model_fn(batch, x)
        recon = display_volume(logits, loss_name)
        diff = recon - x
        mse_sum += float(diff.pow(2).sum().detach().cpu())
        mae_sum += float(diff.abs().sum().detach().cpu())
        voxel_count += int(x.numel())
        bce_losses.append(float(F.binary_cross_entropy(recon.clamp(1e-6, 1 - 1e-6), x.clamp(0, 1)).detach().cpu()))

        scores = recon.detach().cpu().flatten(1).numpy()
        targets = x.detach().cpu().flatten(1).numpy()
        n_vox = targets.shape[1]
        for y, s in zip(targets, scores):
            yc = y - y.mean()
            sc = s - s.mean()
            denom = float(np.linalg.norm(yc) * np.linalg.norm(sc))
            if denom > 0:
                map_pearsons.append(float(np.dot(yc, sc) / denom))
            labels = y > target_threshold
            if labels.any() and (~labels).any():
                voxel_aurocs.append(float(roc_auc_score(labels, s)))
                voxel_average_precisions.append(float(average_precision_score(labels, s)))
            for frac in top_fracs:
                k = max(1, int(round(frac * n_vox)))
                true_top = np.argpartition(y, -k)[-k:]
                pred_top = np.argpartition(s, -k)[-k:]
                overlap = len(set(true_top.tolist()).intersection(pred_top.tolist()))
                top_dice[f'top_{int(frac * 100)}pct_dice'].append(float(2 * overlap / (len(true_top) + len(pred_top))))

    out = {
        'split': split_name,
        'n_maps': int(len(split_ds)),
        'generation_mse': float(mse_sum / max(1, voxel_count)),
        'generation_mae': float(mae_sum / max(1, voxel_count)),
        'generation_bce': float(np.mean(bce_losses)) if bce_losses else None,
        'voxel_auroc': float(np.mean(voxel_aurocs)) if voxel_aurocs else None,
        'voxel_average_precision': float(np.mean(voxel_average_precisions)) if voxel_average_precisions else None,
        'spatial_correlation': float(np.mean(map_pearsons)) if map_pearsons else None,
        'map_pearson': float(np.mean(map_pearsons)) if map_pearsons else None,
        'target_threshold': float(target_threshold),
    }
    out['reconstruction_mse'] = out['generation_mse']
    out['reconstruction_mae'] = out['generation_mae']
    out['reconstruction_bce'] = out['generation_bce']
    out.update({key: float(np.mean(vals)) if vals else None for key, vals in top_dice.items()})
    return out

@torch.no_grad()
def save_volume_examples(model_fn, split_ds, out_dir, name, loss_name, n=4):
    import matplotlib.pyplot as plt
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    batch = next(iter(make_volume_loader(split_ds, n, shuffle=False)))
    x = batch['volume'].float().to(device)
    recon = display_volume(model_fn(batch, x), loss_name).detach().cpu()
    x = x.detach().cpu()
    n = min(n, x.shape[0])
    z = x.shape[-1] // 2
    fig, axes = plt.subplots(3, n, figsize=(3 * n, 8), squeeze=False)
    for i in range(n):
        for row, arr, title in [
            (0, x[i, 0, :, :, z], 'true'),
            (1, recon[i, 0, :, :, z], 'generated'),
            (2, (x[i, 0, :, :, z] - recon[i, 0, :, :, z]).abs(), 'abs diff'),
        ]:
            ax = axes[row, i]
            ax.imshow(arr.T, origin='lower', cmap='magma' if row < 2 else 'viridis')
            ax.set_title(f'{title} {i}')
            ax.axis('off')
    fig.tight_layout()
    path = out_dir / f'{name}_examples.png'
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return str(path)

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(payload, f, indent=2)
    return str(path)

## 5. Stage 1 - CNN Autoencoder

In [ ]:
def resolve_autoencoder_checkpoint():
    patterns = [
        f'{RUNS_DIR}/stage1_autoencoder_atlas_free_*/checkpoints/best_cnn_autoencoder.pt',
        f'{DRIVE_ROOT}/runs_ale_3dcnn_autoencoder_pretrain/autoencoder_atlas_free_*/checkpoints/best_cnn_autoencoder.pt',
    ]
    return find_latest_checkpoint(patterns)

def instantiate_autoencoder_from_config(cfg=None):
    cfg = cfg or {}
    return ALE3DCNNAutoEncoder(
        output_shape=tuple(cfg.get('input_shape', ds.input_shape)),
        base_channels=int(cfg.get('base_channels', BASE_CHANNELS)),
        num_blocks=int(cfg.get('num_blocks', NUM_BLOCKS)),
        latent_dim=int(cfg.get('latent_dim', LATENT_DIM)),
        dropout=float(cfg.get('dropout', DROPOUT)),
        norm=cfg.get('norm', NORM),
        pooling=cfg.get('pooling', POOLING),
    )

def load_autoencoder_checkpoint(path):
    state = torch.load(path, map_location=device, weights_only=False)
    model = instantiate_autoencoder_from_config(state.get('config', {})).to(device)
    model.load_state_dict(state['model'])
    model.eval()
    return model, state

def train_autoencoder_stage1():
    existing = resolve_autoencoder_checkpoint()
    if existing is not None:
        print('Loading existing Stage 1 autoencoder:', existing)
        model, state = load_autoencoder_checkpoint(existing)
        return model, state, existing, str(Path(existing).parents[1])
    if not TRAIN_AUTOENCODER_IF_MISSING:
        raise FileNotFoundError('No Stage 1 checkpoint found and TRAIN_AUTOENCODER_IF_MISSING=False')

    out_dir = Path(RUNS_DIR) / f'stage1_autoencoder_atlas_free_{RUN_STAMP}'
    ckpt_dir = out_dir / 'checkpoints'
    plot_dir = out_dir / 'plots'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)
    config = {
        'stage': 'stage1_cnn_autoencoder', 'input_shape': ds.input_shape,
        'latent_dim': LATENT_DIM, 'base_channels': BASE_CHANNELS, 'num_blocks': NUM_BLOCKS,
        'dropout': DROPOUT, 'norm': NORM, 'pooling': POOLING,
        'loss': AE_LOSS, 'epochs': AE_EPOCHS, 'batch_size': AE_BATCH_SIZE,
        'lr': AE_LR, 'weight_decay': AE_WEIGHT_DECAY,
        'preprocess_config': cache_payload['config'], 'cache_metadata': cache_payload['metadata'],
    }
    write_json(out_dir / 'autoencoder_config.json', config)
    model = instantiate_autoencoder_from_config(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    train_loader = make_volume_loader(train_ds, AE_BATCH_SIZE, shuffle=True)
    val_loader = make_volume_loader(val_ds, AE_BATCH_SIZE, shuffle=False)
    history = {'train_loss': [], 'val_loss': [], 'epoch_time_sec': [], 'lr': []}
    best_loss = float('inf')
    best_state = None
    for epoch in range(AE_EPOCHS):
        start = time.perf_counter()
        model.train()
        losses = []
        for batch in train_loader:
            x = batch['volume'].float().to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                logits = model(x)
                loss = reconstruction_loss(logits, x, AE_LOSS)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        with torch.no_grad():
            val_losses = []
            model.eval()
            for batch in val_loader:
                x = batch['volume'].float().to(device, non_blocking=True)
                val_losses.append(float(reconstruction_loss(model(x), x, AE_LOSS).detach().cpu()))
        val_loss = float(np.mean(val_losses))
        history['train_loss'].append(float(np.mean(losses)))
        history['val_loss'].append(val_loss)
        history['epoch_time_sec'].append(float(time.perf_counter() - start))
        history['lr'].append(float(optimizer.param_groups[0]['lr']))
        print(f'Stage1 epoch {epoch:03d}/{AE_EPOCHS} train={history["train_loss"][-1]:.6f} val={val_loss:.6f}')
        state = {'encoder': model.encoder.state_dict(), 'decoder': model.decoder.state_dict(), 'model': model.state_dict(), 'epoch': epoch, 'val_loss': val_loss, 'history': history, 'config': config}
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(state)
            torch.save(best_state, ckpt_dir / 'best_cnn_autoencoder.pt')
            save_volume_examples(lambda batch, x: model(x), val_ds, plot_dir, 'autoencoder_reconstruction', AE_LOSS)
    torch.save(state, ckpt_dir / 'last_cnn_autoencoder.pt')
    write_json(out_dir / 'autoencoder_training_history.json', history)
    model.load_state_dict(best_state['model'])
    model.eval()
    return model, best_state, str(ckpt_dir / 'best_cnn_autoencoder.pt'), str(out_dir)

stage1_autoencoder, STAGE1_STATE, BEST_AE_CHECKPOINT, STAGE1_RUN_DIR = train_autoencoder_stage1()
stage1_metrics = {
    'model': 'atlas_free_ale3dcnn_autoencoder',
    'checkpoint': BEST_AE_CHECKPOINT,
    'metrics': [
        volume_pair_metrics(lambda batch, x: stage1_autoencoder(x), val_ds, split_name='val', loss_name=AE_LOSS, batch_size=AE_BATCH_SIZE),
        volume_pair_metrics(lambda batch, x: stage1_autoencoder(x), test_ds, split_name='test', loss_name=AE_LOSS, batch_size=AE_BATCH_SIZE),
    ],
}
write_json(Path(STAGE1_RUN_DIR) / 'autoencoder_reconstruction_metrics.json', stage1_metrics)
pd.DataFrame(stage1_metrics['metrics']).to_csv(Path(STAGE1_RUN_DIR) / 'autoencoder_reconstruction_metrics.csv', index=False)
save_volume_examples(lambda batch, x: stage1_autoencoder(x), test_ds, Path(STAGE1_RUN_DIR) / 'plots', 'autoencoder_test_reconstruction', AE_LOSS)
pd.DataFrame(stage1_metrics['metrics'])

## 6. Stage 2 - Text to CNN Latent and Frozen Decoder

In [ ]:
def resolve_stage2_checkpoint(init):
    patterns = [f'{RUNS_DIR}/stage2_text_to_cnn_latent_{init}_*/checkpoints/cnn_text_mse_or_bce_projection.pt']
    return find_latest_checkpoint(patterns)

def train_text_decoder_stage2(init):
    existing = resolve_stage2_checkpoint(init)
    if existing is not None:
        print(f'Loading existing Stage 2 projection ({init}):', existing)
        return existing, torch.load(existing, map_location='cpu', weights_only=False), str(Path(existing).parents[1])
    if not TRAIN_TEXT_DECODER_IF_MISSING:
        raise FileNotFoundError(f'No Stage 2 {init} checkpoint and TRAIN_TEXT_DECODER_IF_MISSING=False')

    out_dir = Path(RUNS_DIR) / f'stage2_text_to_cnn_latent_{init}_{RUN_STAMP}'
    ckpt_dir = out_dir / 'checkpoints'
    plot_dir = out_dir / 'plots'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    decoder = copy.deepcopy(stage1_autoencoder.decoder).to(device).eval()
    for p in decoder.parameters():
        p.requires_grad_(False)
    text_proj = make_text_projector(init, freeze=False).to(device)
    optimizer = torch.optim.AdamW(text_proj.parameters(), lr=TEXT_DECODER_LR, weight_decay=TEXT_DECODER_WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    train_loader = make_volume_loader(train_ds, TEXT_DECODER_BATCH_SIZE, shuffle=True)
    val_loader = make_volume_loader(val_ds, TEXT_DECODER_BATCH_SIZE, shuffle=False)
    config = {
        'stage': 'stage2_text_to_cnn_latent_decoder', 'init': init,
        'loss': TEXT_DECODER_LOSS, 'epochs': TEXT_DECODER_EPOCHS, 'batch_size': TEXT_DECODER_BATCH_SIZE,
        'lr': TEXT_DECODER_LR, 'weight_decay': TEXT_DECODER_WEIGHT_DECAY,
        'stage1_checkpoint': BEST_AE_CHECKPOINT,
    }
    write_json(out_dir / 'text_decoder_config.json', config)
    history = {'train_loss': [], 'val_loss': [], 'epoch_time_sec': []}
    best_loss = float('inf')
    best_state = None
    for epoch in range(TEXT_DECODER_EPOCHS):
        start = time.perf_counter()
        text_proj.train()
        losses = []
        for batch in train_loader:
            x = batch['volume'].float().to(device, non_blocking=True)
            text = batch['text'].float().to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                latent = text_proj(text)
                logits = decoder(latent)
                loss = reconstruction_loss(logits, x, TEXT_DECODER_LOSS)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(text_proj.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        text_proj.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                x = batch['volume'].float().to(device, non_blocking=True)
                text = batch['text'].float().to(device, non_blocking=True)
                val_losses.append(float(reconstruction_loss(decoder(text_proj(text)), x, TEXT_DECODER_LOSS).detach().cpu()))
        val_loss = float(np.mean(val_losses))
        history['train_loss'].append(float(np.mean(losses)))
        history['val_loss'].append(val_loss)
        history['epoch_time_sec'].append(float(time.perf_counter() - start))
        print(f'Stage2 {init} epoch {epoch:03d}/{TEXT_DECODER_EPOCHS} train={history["train_loss"][-1]:.6f} val={val_loss:.6f}')
        state = {
            'text_proj': text_proj.state_dict(), 'decoder': decoder.state_dict(),
            'epoch': epoch, 'val_loss': val_loss, 'history': history, 'config': config,
        }
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(state)
            torch.save(best_state, ckpt_dir / 'cnn_text_mse_or_bce_projection.pt')
            save_volume_examples(lambda batch, x: decoder(text_proj(batch['text'].float().to(device))), val_ds, plot_dir, f'text_to_brain_{init}', TEXT_DECODER_LOSS)
    write_json(out_dir / 'text_decoder_training_history.json', history)
    text_proj.load_state_dict(best_state['text_proj'])
    text_proj.eval()
    metrics = {
        'model': f'cnn_text_to_brain_{init}',
        'checkpoint': str(ckpt_dir / 'cnn_text_mse_or_bce_projection.pt'),
        'metrics': [
            volume_pair_metrics(lambda batch, x: decoder(text_proj(batch['text'].float().to(device))), val_ds, split_name='val', loss_name=TEXT_DECODER_LOSS, batch_size=TEXT_DECODER_BATCH_SIZE),
            volume_pair_metrics(lambda batch, x: decoder(text_proj(batch['text'].float().to(device))), test_ds, split_name='test', loss_name=TEXT_DECODER_LOSS, batch_size=TEXT_DECODER_BATCH_SIZE),
        ],
    }
    write_json(out_dir / 'text_to_brain_generation_metrics.json', metrics)
    pd.DataFrame(metrics['metrics']).to_csv(out_dir / 'text_to_brain_generation_metrics.csv', index=False)
    return str(ckpt_dir / 'cnn_text_mse_or_bce_projection.pt'), best_state, str(out_dir)

STAGE2_RUNS = {}
for init in TEXT_DECODER_INITS:
    ckpt, state, run_dir = train_text_decoder_stage2(init)
    STAGE2_RUNS[init] = {'checkpoint': ckpt, 'state': state, 'run_dir': run_dir}
pd.DataFrame([{k: v for k, v in row.items() if k != 'state'} for row in STAGE2_RUNS.values()])

## 7. Stage 3 - Contrastive Alignment

In [ ]:
class FrozenAwareALETrainer(train_ale_cnn.ALETrainer):
    def __init__(self, brain_encoder, text_proj, args, device, freeze_brain_encoder=False):
        self.freeze_brain_encoder = freeze_brain_encoder
        if freeze_brain_encoder:
            for p in brain_encoder.parameters():
                p.requires_grad_(False)
        super().__init__(brain_encoder, text_proj, args, device)

    def _forward(self, batch):
        if self.freeze_brain_encoder:
            self.brain_encoder.eval()
        return super()._forward(batch)

def make_text_projector_for_contrastive(init_name, freeze):
    if init_name == 'pretrained_infonce':
        proj = make_text_projector('pretrained_infonce', freeze=freeze)
    elif init_name.startswith('stage2_'):
        stage2_init = init_name.replace('stage2_', '')
        proj = make_text_projector(STAGE2_RUNS[stage2_init]['state']['config']['init'], freeze=False)
        proj.load_state_dict(STAGE2_RUNS[stage2_init]['state']['text_proj'])
        for p in proj.parameters():
            p.requires_grad_(not freeze)
    else:
        raise ValueError(init_name)
    return proj

def contrastive_args_for(run_dir, variant, freeze_text):
    return argparse.Namespace(
        mode='atlas_free', model='ale_3dcnn', epochs=CONTRASTIVE_EPOCHS,
        batch_size=CONTRASTIVE_BATCH_SIZE, batch_size_auto=False, grad_accum_steps=1,
        lr_cnn=LR_CNN, lr_proj=LR_PROJ, warmup_epochs=5, temperature=TEMPERATURE,
        val_interval=VAL_INTERVAL, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        monitor_metric='paper_recall_curve_auc', base_channels=BASE_CHANNELS,
        num_blocks=NUM_BLOCKS, out_dim=LATENT_DIM, dropout=DROPOUT, norm=NORM, pooling=POOLING,
        mlp_hidden_dim=1024, kernel_fwhm_mm=KERNEL_FWHM_MM, resolution_mm=RESOLUTION_MM,
        crop_to_brain=CROP_TO_BRAIN, normalize=NORMALIZE, no_clamp=not CLAMP,
        cache_dtype=CACHE_DTYPE, cache_file=ATLAS_FREE_CACHE, force_rebuild_cache=False,
        build_cache_only=False, max_papers=None, text_proj_init='custom', freeze_text_proj=freeze_text,
        device='cuda' if torch.cuda.is_available() else 'auto', amp=True, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(), seed=SEED, val_frac=0.1, test_frac=0.1,
        checkpoint_dir=str(Path(run_dir) / 'checkpoints'), run_dir=str(run_dir), comparison_file=COMPARISON_FILE,
        save_plots=True, umap=True, eval_neurovlm_baseline=False, semantic_eval=True,
        eval_resource_dir=EVAL_RESOURCE_DIR, mesh_json=None, train_sanity_n=TRAIN_SANITY_N,
        variant=variant,
    )

def preflight_batch_size(brain_encoder, text_proj, freeze_brain=False):
    candidates = sorted(set(CONTRASTIVE_BATCH_SIZE_CANDIDATES + [CONTRASTIVE_BATCH_SIZE]), reverse=True)
    loss_fn = InfoNCELoss(TEMPERATURE)
    params = [p for p in list(brain_encoder.parameters()) + list(text_proj.parameters()) if p.requires_grad]
    if not params:
        raise ValueError('No trainable parameters for variant.')
    for bs in candidates:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            batch = next(iter(DataLoader(train_ds, batch_size=min(bs, len(train_ds)), shuffle=False, num_workers=0)))
            x = batch['volume'].float().to(device)
            text = batch['text'].float().to(device)
            brain_encoder.to(device); text_proj.to(device)
            for p in params:
                p.grad = None
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                loss = loss_fn(brain_encoder(x), text_proj(text))
            loss.backward()
            peak = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
            for p in params:
                p.grad = None
            del batch, x, text, loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f'Preflight selected batch_size={bs} peak_vram={peak:.0f}MB')
            return int(bs), float(peak)
        except RuntimeError as exc:
            if 'out of memory' not in str(exc).lower():
                raise
            print(f'Preflight batch_size={bs} OOM; trying smaller.')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise RuntimeError('No contrastive batch size fit.')

def run_contrastive_variant(variant):
    name = variant['name']
    run_dir = Path(RUNS_DIR) / f'stage3_contrastive_{name}_{RUN_STAMP}'
    summary_path = run_dir / 'main_comparison_summary_row.csv'
    if summary_path.exists():
        print('Using existing Stage 3 run:', run_dir)
        return {'variant': name, 'run_dir': str(run_dir), 'summary': str(summary_path), 'loaded_existing': True}
    run_dir.mkdir(parents=True, exist_ok=True)
    brain_state = STAGE1_STATE['encoder'] if variant.get('encoder_init') == 'stage1_autoencoder' else None
    brain_encoder = build_cnn_encoder(load_state=brain_state, freeze=variant['freeze_brain'])
    text_proj = make_text_projector_for_contrastive(variant['text_init'], freeze=variant['freeze_text'])
    batch_size, preflight_peak = preflight_batch_size(brain_encoder, text_proj, freeze_brain=variant['freeze_brain'])
    args = contrastive_args_for(run_dir, name, variant['freeze_text'])
    args.batch_size = batch_size
    args.preflight_peak_vram_mb = preflight_peak
    write_json(run_dir / 'config.json', {**vars(args), **variant})
    write_json(run_dir / 'preprocessing_config.json', {**cache_payload['config'], 'metadata': cache_payload['metadata']})
    trainer = FrozenAwareALETrainer(brain_encoder, text_proj, args, device, freeze_brain_encoder=variant['freeze_brain'])
    trainer.fit(train_ds, val_ds)
    trainer.restore_best()
    all_metrics, eval_payloads = {}, {}
    if args.train_sanity_n and args.train_sanity_n > 0:
        train_metrics, train_brain, train_text = train_ale_cnn.evaluate_train_subset_sanity(trainer, train_ds, args.train_sanity_n)
        all_metrics['train_subset'] = train_metrics
        eval_payloads['train_subset'] = (train_metrics, train_brain, train_text)
    for split_name, split_ds in [('val', val_ds), ('test', test_ds)]:
        metrics, brain_emb, text_emb = trainer.evaluate(split_ds)
        all_metrics[split_name] = metrics
        eval_payloads[split_name] = (metrics, brain_emb, text_emb)
        print(f'{name} {split_name}: AUC={metrics["paper_recall_curve_auc"]:.4f} r@10={metrics["mean_recall@10"]:.4f}')
    write_json(run_dir / 'training_history.json', trainer.history)
    write_json(run_dir / 'eval_results.json', all_metrics)
    for split_name, (_, brain_emb, text_emb) in eval_payloads.items():
        train_ale_cnn.recall_curve_frame(text_emb, brain_emb).to_csv(run_dir / f'{split_name}_recall_curve.csv', index=False)
        write_json(run_dir / f'{split_name}_recall_curve.json', train_ale_cnn.recall_curve_payload(text_emb, brain_emb))
    test_metrics, test_brain, test_text = eval_payloads['test']
    cov = test_ds.covariate_frame()
    diag_df = train_ale_cnn.retrieval_diagnostics(test_text, test_brain, cov)
    diag_df.to_csv(run_dir / 'test_retrieval_diagnostics.csv', index=False)
    train_ale_cnn.save_embedding_correlations(run_dir, test_brain, cov)
    train_ale_cnn.save_plots(run_dir, trainer.history, pd.read_csv(run_dir / 'test_recall_curve.csv'), diag_df, test_brain)
    try:
        from experiments.semantic_model_eval import run_ale_network_labeling, run_embedding_semantic_evaluations
        from neurovlm.data import load_masker
        semantic_summary = run_embedding_semantic_evaluations(
            model_name=name, brain_embeddings=test_brain, text_embeddings=test_text,
            raw_text_embeddings=test_ds.raw_text_embeddings, pmids=test_ds.pmids,
            text_projector=trainer.text_proj, out_dir=run_dir, device=device,
            resource_dir=EVAL_RESOURCE_DIR, mesh_json=None,
            extra_summary={**variant, 'actual_batch_size': int(args.batch_size), 'preflight_peak_vram_mb': float(preflight_peak), 'peak_vram_mb': float(max(trainer.history['peak_vram_mb'] or [0.0])), 'training_time_per_epoch': float(np.mean(trainer.history['epoch_time_sec']))},
        )
        network_metrics = run_ale_network_labeling(trainer=trainer, preprocess_config=preprocess_config, masker=load_masker(), out_dir=run_dir, device=device, resource_dir=EVAL_RESOURCE_DIR)
        semantic_summary.update({'network_accuracy': network_metrics.get('accuracy'), 'network_top_2_accuracy': network_metrics.get('top_2_accuracy'), 'network_macro_auc': network_metrics.get('macro_auc')})
        semantic_summary.update({k: v for k, v in network_metrics.items() if k.startswith('network_term_')})
        write_json(run_dir / 'main_comparison_summary_row.json', semantic_summary)
        pd.DataFrame([semantic_summary]).to_csv(run_dir / 'main_comparison_summary_row.csv', index=False)
    except Exception as exc:
        print('WARNING: semantic evaluation failed:', exc)
        print(traceback.format_exc())
    return {'variant': name, 'run_dir': str(run_dir), 'summary': str(run_dir / 'main_comparison_summary_row.csv'), 'trainer': trainer, **variant}

STAGE3_VARIANTS = [
    {'name': 'ae_encoder_frozen_stage2_pretrained_text_trainable', 'encoder_init': 'stage1_autoencoder', 'freeze_brain': True, 'text_init': 'stage2_pretrained_infonce', 'freeze_text': False},
    {'name': 'ae_encoder_finetune_stage2_pretrained_text_frozen', 'encoder_init': 'stage1_autoencoder', 'freeze_brain': False, 'text_init': 'stage2_pretrained_infonce', 'freeze_text': True},
    {'name': 'ae_encoder_finetune_stage2_pretrained_text_trainable', 'encoder_init': 'stage1_autoencoder', 'freeze_brain': False, 'text_init': 'stage2_pretrained_infonce', 'freeze_text': False},
    {'name': 'ae_encoder_finetune_pretrained_infonce_text_frozen', 'encoder_init': 'stage1_autoencoder', 'freeze_brain': False, 'text_init': 'pretrained_infonce', 'freeze_text': True},
    {'name': 'ae_encoder_finetune_pretrained_infonce_text_trainable', 'encoder_init': 'stage1_autoencoder', 'freeze_brain': False, 'text_init': 'pretrained_infonce', 'freeze_text': False},
    {'name': 'scratch_encoder_pretrained_infonce_text_trainable', 'encoder_init': 'scratch', 'freeze_brain': False, 'text_init': 'pretrained_infonce', 'freeze_text': False},
]
STAGE3_RUNS = [run_contrastive_variant(v) for v in STAGE3_VARIANTS]
pd.DataFrame([{k: v for k, v in row.items() if k != 'trainer'} for row in STAGE3_RUNS])

## 8. Stage 4 - Text-to-Brain Generation Evaluation

In [ ]:
def evaluate_projection_generation(name, text_proj_state, init_for_arch, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    decoder = copy.deepcopy(stage1_autoencoder.decoder).to(device).eval()
    for p in decoder.parameters():
        p.requires_grad_(False)
    proj = make_text_projector(init_for_arch, freeze=True).to(device)
    proj.load_state_dict(text_proj_state)
    proj.eval()
    model_fn = lambda batch, x: decoder(proj(batch['text'].float().to(device)))
    metrics = {
        'model': name,
        'metrics': [
            volume_pair_metrics(model_fn, val_ds, split_name='val', loss_name=TEXT_DECODER_LOSS, batch_size=TEXT_DECODER_BATCH_SIZE),
            volume_pair_metrics(model_fn, test_ds, split_name='test', loss_name=TEXT_DECODER_LOSS, batch_size=TEXT_DECODER_BATCH_SIZE),
        ],
    }
    write_json(out_dir / 'text_to_brain_generation_metrics.json', metrics)
    pd.DataFrame(metrics['metrics']).to_csv(out_dir / 'text_to_brain_generation_metrics.csv', index=False)
    save_volume_examples(model_fn, test_ds, out_dir / 'plots', name, TEXT_DECODER_LOSS)
    return metrics

GENERATION_RUNS = []
for init, payload in STAGE2_RUNS.items():
    GENERATION_RUNS.append(evaluate_projection_generation(
        f'stage2_text_to_brain_{init}',
        payload['state']['text_proj'],
        payload['state']['config']['init'],
        Path(payload['run_dir']) / 'stage4_generation_eval',
    ))

for row in STAGE3_RUNS:
    trainer = row.get('trainer')
    if trainer is None:
        continue
    init_for_arch = 'pretrained_infonce' if row['text_init'] in ['pretrained_infonce', 'stage2_pretrained_infonce'] else 'random'
    GENERATION_RUNS.append(evaluate_projection_generation(
        f'stage3_generation_{row["variant"]}',
        trainer.text_proj.state_dict(),
        init_for_arch,
        Path(row['run_dir']) / 'stage4_generation_eval',
    ))

flat_rows = []
for payload in GENERATION_RUNS:
    for metrics in payload['metrics']:
        flat_rows.append({'model': payload['model'], **metrics})
generation_df = pd.DataFrame(flat_rows)
generation_df.to_csv(Path(RUNS_DIR) / f'generation_summary_{RUN_STAMP}.csv', index=False)
generation_df

## 9. Final Comparison Tables

In [ ]:
IMPORTANT_RETRIEVAL = [
    'model', 'variant', 'network_term_recall@5', 'network_term_recall@10', 'network_term_mrr',
    'mesh_recall@10', 'semantic_recall@10', 'semantic_recall@50',
    'semantic_paper_style_recall_curve_auc', 'exact_pmid_paper_recall_curve_auc',
    'actual_batch_size', 'peak_vram_mb', 'training_time_per_epoch',
]

def load_summary(path):
    path = Path(path)
    if path.is_dir():
        path = path / 'main_comparison_summary_row.csv'
    if not path.exists():
        return None
    return pd.read_csv(path).iloc[0].to_dict()

retrieval_rows = []
for row in STAGE3_RUNS:
    summary = load_summary(row['run_dir'])
    if summary:
        summary['variant'] = row['variant']
        retrieval_rows.append(summary)

for p in [
    Path(REPO_DIR) / 'experiments/runs/neurovlm_pubmed_semantic_baseline',
    Path(DRIVE_ROOT) / 'runs/neurovlm_pubmed_semantic_baseline',
    Path('/content/drive/MyDrive/neurovlm/runs/neurovlm_pubmed_semantic_baseline'),
]:
    summary = load_summary(p)
    if summary:
        summary['variant'] = 'pretrained_neurovlm_mlp_baseline'
        retrieval_rows.append(summary)
        break

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df.to_csv(Path(RUNS_DIR) / f'retrieval_summary_{RUN_STAMP}.csv', index=False)
cols = [c for c in IMPORTANT_RETRIEVAL if c in retrieval_df.columns]
retrieval_df[cols]